# Importing required packages

In [1]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

In [3]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import cellcharter as cc
import matplotlib.pyplot as plt
import yaml
import squidpy as sq
from pathlib import Path

# Required inputs

In [4]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Set up directories
metadata_dir = Path(config['metadata_dir'])
results_dir = Path(config['results_dir'])

In [5]:
#read the adata object
adata = ad.read(results_dir / 'adata.h5ad')

/Users/jawadalaaedeen/miniconda3/envs/cellcharter-env/lib/python3.10/site-packages/anndata/__init__.py:42: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(
/Users/jawadalaaedeen/miniconda3/envs/cellcharter-env/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [ ]:
#Let's try with Run145
adata_select = adata[adata.obs['run'].isin([145,141])]
#Keep only the columns where not all values are NA
valid_vars = ~np.all(np.isnan(adata_select.X), axis=0)
adata_select = adata_select[:, valid_vars]

# Adata concatenation

In [4]:
#conctenate all the adata files
adata_list = []
for roi in roi_list:
    adata = ad.read_h5ad(roi / "adata.h5ad")
    adata_list.append(adata)
adata = ad.concat(adata_list)
#make obs names unique
adata.obs_names_make_unique()
adata.obs.run = adata.obs.run.astype("category")
adata.obs.patient_ID = adata.obs.patient_ID.astype("category")
adata.raw = adata.copy()
#do variance stabilizing transformation
adata.X = np.arcsinh(adata.X)

/opt/anaconda3/envs/cellcharter_env/lib/python3.9/site-packages/anndata/_core/merge.py:1284: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concat_annot = pd.concat(
/opt/anaconda3/envs/cellcharter_env/lib/python3.9/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


# Adata scaling

In [5]:
#scale the data
sc.pp.scale(adata)
adata.X = adata.X.astype(np.float32).copy()

# Training TRVAE

In [7]:
#Training my own model 
condition_key = 'patient_ID'
cell_type_key = 'cell_type'
conditions = adata.obs[condition_key].unique().tolist()


trvae_epochs = 500
surgery_epochs = 500

early_stopping_kwargs = {
    "early_stopping_metric": "val_unweighted_loss",
    "threshold": 0,
    "patience": 20,
    "reduce_lr": True,
    "lr_patience": 13,
    "lr_factor": 0.1,
}
trvae = cc.tl.TRVAE(
    adata=adata,
    condition_key=condition_key,
    conditions=conditions,
    recon_loss='mse',
    use_mmd=False,
)
trvae.train(
    n_epochs=trvae_epochs,
    alpha_epoch_anneal=200,
    early_stopping_kwargs=early_stopping_kwargs,
    enable_progress_bar=True,
    
)
trvae.save('trvae', overwrite=True)


INITIALIZING NEW NETWORK..............
Encoder Architecture:
	Input Layer in, out and cond: 61 256 10
	Hidden Layer 1 in/out: 256 64
	Mean/Var Layer in/out: 64 10
Decoder Architecture:
	First Layer in, out and cond:  10 64 10
	Hidden Layer 1 in/out: 64 256
	Output Layer in/out:  256 61 

Preparing (428089, 61)
Instantiating dataset
 |█████---------------| 25.4%  - val_loss: 15.2799879416 - val_recon_loss: 11.5783321523 - val_kl_loss: 5.87564415011
ADJUSTED LR
 |██████--------------| 31.4%  - val_loss: 15.4571542085 - val_recon_loss: 11.4295224432 - val_kl_loss: 5.1636306464
ADJUSTED LR
 |███████-------------| 39.4%  - val_loss: 16.1748278376 - val_recon_loss: 11.6182279701 - val_kl_loss: 4.6495916025
ADJUSTED LR
 |████████------------| 40.8%  - val_loss: 16.2362429889 - val_recon_loss: 11.5690821918 - val_kl_loss: 4.6671608170
Stopping early: no improvement of more than 0 nats in 20 epochs
If the early stopping criterion is too strong, please instantiate it with different parameters i